# Baby-care Insight ?????? Notebook

? Notebook ??????????????????????????????????????????????????????????

??????**???????? dashboard.html**???????????????????????

In [ ]:
# 1. ???????
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

from PIL import Image
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRADE_FILE = DATA_RAW_DIR / "sam_tianchi_mum_baby_trade_history.csv"
BABY_FILE = DATA_RAW_DIR / "sam_tianchi_mum_baby.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_showcase"
PROCESSED_DIR = OUTPUT_DIR / "processed"
TABLE_DIR = OUTPUT_DIR / "tables"
CHART_DIR = OUTPUT_DIR / "charts"
MODEL_DIR = OUTPUT_DIR / "model_results"

for d in [OUTPUT_DIR, PROCESSED_DIR, TABLE_DIR, CHART_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("??????", PROJECT_ROOT)
print("?????", TRADE_FILE.exists(), TRADE_FILE)
print("?????", BABY_FILE.exists(), BABY_FILE)
print("???????", OUTPUT_DIR)

In [ ]:
# 2. ???????????
REPORT_COLORS = ["#4E79A7", "#F28E2B", "#59A14F", "#E15759", "#76B7B2", "#B07AA1"]
CHART_COLORS = ["#3B82F6", "#22C55E", "#F59E0B", "#8B5CF6", "#06B6D4", "#EF476F", "#64748B"]


def setup_chinese_font():
    font_candidates = [
        "Microsoft YaHei", "SimHei", "WenQuanYi Micro Hei", "WenQuanYi Zen Hei",
        "Noto Sans CJK SC", "Arial Unicode MS",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    chosen = next((f for f in font_candidates if f in available), "DejaVu Sans")
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = [chosen, "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid", palette=REPORT_COLORS)
    print("???????", chosen)
    return chosen

FONT_NAME = setup_chinese_font()


def polish_axes(ax, grid_axis="y"):
    ax.set_facecolor("white")
    ax.grid(True, axis=grid_axis, color="#E5E7EB", linewidth=0.8, alpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#D0D5DD")
    ax.spines["bottom"].set_color("#D0D5DD")
    ax.tick_params(colors="#344054", labelsize=10)

In [ ]:
# 3. ????????
CAT1_NAME_MAP = {
    "28": "????",
    "50014815": "????",
    "50008168": "????",
    "38": "????",
    "50022520": "????",
    "122650008": "????",
}

CAT1_WORDS_EXTENDED = {
    "????": ["??", "??", "??", "????", "??", "???", "??", "????", "???", "???", "???", "???", "???", "??", "???", "????", "????"],
    "????": ["???", "???", "???", "??", "??", "???", "???", "???", "????", "???", "?????"],
    "????": ["???", "??", "???", "??", "??", "??", "??", "??", "??", "???", "???", "??", "??", "??", "????"],
    "????": ["????", "???", "????", "????", "???", "????", "????", "???", "???"],
    "????": ["??", "??", "????", "????", "???", "??", "??", "??", "??", "????", "????", "???"],
    "????": ["???", "??", "??", "????", "??", "??", "??", "??", "???", "????", "??"],
}


def normalize_cat1(value):
    if pd.isna(value):
        return "unknown"
    try:
        return str(int(float(value)))
    except Exception:
        return str(value)


def cat1_to_name(value):
    return CAT1_NAME_MAP.get(normalize_cat1(value), "??????")

In [ ]:
# 4. ????
trade_df = pd.read_csv(TRADE_FILE)
baby_df = pd.read_csv(BABY_FILE)

print("???????", len(trade_df))
print("????????", len(baby_df))
print("??????", trade_df.columns.tolist())
print("???????", baby_df.columns.tolist())

display(trade_df.head())
display(baby_df.head())

In [ ]:
# 5. ??????
quality_overview = pd.DataFrame([
    {"dataset": "trade_history", "rows": len(trade_df), "columns": trade_df.shape[1], "duplicate_rows": trade_df.duplicated().sum()},
    {"dataset": "baby_info", "rows": len(baby_df), "columns": baby_df.shape[1], "duplicate_rows": baby_df.duplicated().sum()},
])

missing_values = pd.concat([
    trade_df.isna().sum().reset_index().rename(columns={"index": "field", 0: "missing_count"}).assign(dataset="trade_history"),
    baby_df.isna().sum().reset_index().rename(columns={"index": "field", 0: "missing_count"}).assign(dataset="baby_info"),
], ignore_index=True)

quality_overview.to_csv(TABLE_DIR / "01_data_overview.csv", index=False, encoding="utf-8-sig")
missing_values.to_csv(TABLE_DIR / "02_missing_values.csv", index=False, encoding="utf-8-sig")

print("?????")
display(quality_overview)
print("??????")
display(missing_values)

In [ ]:
# 6. ????????
trade = trade_df.copy().drop_duplicates()
baby = baby_df.copy().drop_duplicates(subset=["user_id"])

trade["property"] = trade["property"].fillna("unknown").astype(str)
trade["day"] = trade["day"].astype(str)
trade["date"] = pd.to_datetime(trade["day"], format="%Y%m%d", errors="coerce")
trade = trade.dropna(subset=["date", "user_id", "auction_id", "cat_id", "cat1", "buy_mount"])
trade = trade[trade["buy_mount"] > 0].copy()

trade["year"] = trade["date"].dt.year
trade["month"] = trade["date"].dt.month
trade["weekday"] = trade["date"].dt.weekday + 1
trade["year_month"] = trade["date"].dt.to_period("M").astype(str)
trade["cat1_name"] = trade["cat1"].apply(cat1_to_name)

baby["birthday"] = pd.to_datetime(baby["birthday"].astype(str), format="%Y%m%d", errors="coerce")
baby["gender"] = baby["gender"].map({0: "female", 1: "male", 2: "unknown"}).fillna("unknown")

merged_df = trade.merge(baby[["user_id", "birthday", "gender"]], on="user_id", how="left")
merged_df["gender"] = merged_df["gender"].fillna("unknown")
merged_df["baby_age"] = ((merged_df["date"] - merged_df["birthday"]).dt.days / 365.25).round(2)
merged_df.loc[(merged_df["baby_age"] < 0) | (merged_df["baby_age"] > 8), "baby_age"] = np.nan
merged_df["baby_age"] = merged_df["baby_age"].fillna(-1)

bins = [-2, 0, 0.5, 1, 2, 3, 8]
labels = ["unknown", "0-6??", "6-12??", "1-2?", "2-3?", "3???"]
merged_df["baby_age_group"] = pd.cut(merged_df["baby_age"], bins=bins, labels=labels, include_lowest=True).astype(str)

merged_df.to_csv(PROCESSED_DIR / "merged_mother_baby_data.csv", index=False, encoding="utf-8-sig")
print("??????????", merged_df.shape)
display(merged_df.head())

In [ ]:
# 7. ??????
category_sales = merged_df.groupby("cat1_name", as_index=False).agg(
    total_buy_mount=("buy_mount", "sum"),
    trade_count=("buy_mount", "size"),
    user_count=("user_id", "nunique"),
).sort_values("total_buy_mount", ascending=False)

monthly_sales = merged_df.groupby("year_month", as_index=False).agg(
    total_buy_mount=("buy_mount", "sum"),
    trade_count=("buy_mount", "size"),
)

weekday_sales = merged_df.groupby("weekday", as_index=False).agg(
    total_buy_mount=("buy_mount", "sum"),
    trade_count=("buy_mount", "size"),
)

profile_summary = merged_df.groupby(["gender", "baby_age_group"], as_index=False).agg(
    trade_count=("buy_mount", "size"),
    total_buy_mount=("buy_mount", "sum"),
)

category_sales.to_csv(TABLE_DIR / "03_category_sales_summary.csv", index=False, encoding="utf-8-sig")
monthly_sales.to_csv(TABLE_DIR / "04_monthly_sales_summary.csv", index=False, encoding="utf-8-sig")
weekday_sales.to_csv(TABLE_DIR / "05_weekday_sales_summary.csv", index=False, encoding="utf-8-sig")
profile_summary.to_csv(TABLE_DIR / "06_user_profile_summary.csv", index=False, encoding="utf-8-sig")

display(category_sales)
display(monthly_sales.tail())
display(weekday_sales)

In [ ]:
# 8. ????????????????????
model_df = merged_df.copy()
model_df["property"] = model_df["property"].fillna("unknown").astype(str)
model_df["gender"] = model_df["gender"].fillna("unknown").astype(str)
model_df["baby_age_group"] = model_df["baby_age_group"].fillna("unknown").astype(str)

cat1_features = ["cat_id", "property", "auction_id", "month", "weekday", "gender", "baby_age", "baby_age_group"]
y_cat1 = model_df["cat1"].astype(str)
X_cat1_raw = model_df[cat1_features].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_cat1_raw, y_cat1, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cat1
)

# ???????????????????
cat_id_freq = X_train_raw["cat_id"].value_counts(normalize=True)
property_freq = X_train_raw["property"].value_counts(normalize=True)
auction_freq = X_train_raw["auction_id"].value_counts(normalize=True)

cat_id_encoder = LabelEncoder().fit(X_train_raw["cat_id"].astype(str))
gender_encoder = LabelEncoder().fit(pd.concat([X_train_raw["gender"], pd.Series(["unknown"])]).astype(str))
age_group_encoder = LabelEncoder().fit(pd.concat([X_train_raw["baby_age_group"], pd.Series(["unknown"])]).astype(str))


def safe_label_transform(encoder, values):
    known = set(encoder.classes_)
    values = values.astype(str).where(values.astype(str).isin(known), "unknown")
    if "unknown" not in known:
        # ???????? unknown???????????????????????
        values = values.where(values.isin(known), encoder.classes_[0])
    return encoder.transform(values.astype(str))


def build_cat1_features(df):
    out = pd.DataFrame(index=df.index)
    out["cat_id_encoded"] = safe_label_transform(cat_id_encoder, df["cat_id"].astype(str))
    out["cat_id_freq"] = df["cat_id"].map(cat_id_freq).fillna(0)
    out["property_freq"] = df["property"].map(property_freq).fillna(0)
    out["auction_id_freq"] = df["auction_id"].map(auction_freq).fillna(0)
    out["month"] = df["month"].astype(float)
    out["weekday"] = df["weekday"].astype(float)
    out["gender_encoded"] = safe_label_transform(gender_encoder, df["gender"].astype(str))
    out["baby_age"] = df["baby_age"].astype(float)
    out["baby_age_group_encoded"] = safe_label_transform(age_group_encoder, df["baby_age_group"].astype(str))
    return out

X_train = build_cat1_features(X_train_raw)
X_test = build_cat1_features(X_test_raw)

X_train.head()

In [ ]:
# 9. ????cat1 ??????
cat1_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

cat1_metrics = []
cat1_predictions = {}
for name, model in cat1_models.items():
    start = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)
    row = {
        "model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_weighted": precision,
        "Recall_weighted": recall,
        "F1_weighted": f1,
        "Training Time": time.time() - start,
    }
    cat1_metrics.append(row)
    cat1_predictions[name] = pred
    print(name, row)

cat1_metrics_df = pd.DataFrame(cat1_metrics).sort_values("F1_weighted", ascending=False)
cat1_metrics_df.to_csv(MODEL_DIR / "cat1_model_metrics.csv", index=False, encoding="utf-8-sig")

best_cat1_name = cat1_metrics_df.iloc[0]["model"]
show_model_name = "Random Forest"
print("??????", best_cat1_name)
print("???????", show_model_name)
display(cat1_metrics_df)

In [ ]:
# 10. ???????? Random Forest ?????????
rf_model = cat1_models["Random Forest"]
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False)
feature_importance.to_csv(MODEL_DIR / "cat1_feature_importance.csv", index=False, encoding="utf-8-sig")
feature_importance.head(10)

In [ ]:
# 11. ?????high_buy ????????????? buy_mount ????
high_df = model_df.copy()
threshold = high_df["buy_mount"].quantile(0.75)
high_df["high_buy"] = (high_df["buy_mount"] > threshold).astype(int)

high_features = ["month", "weekday", "baby_age", "cat1"]
high_X = high_df[high_features].copy()
high_X["baby_age"] = high_X["baby_age"].fillna(-1)
high_X["cat1"] = high_X["cat1"].astype(str)
high_y = high_df["high_buy"]

Xh_train_raw, Xh_test_raw, yh_train, yh_test = train_test_split(
    high_X, high_y, test_size=0.2, random_state=RANDOM_STATE, stratify=high_y
)

cat1_aux_encoder = LabelEncoder().fit(Xh_train_raw["cat1"].astype(str))

def build_high_features(df):
    out = df.copy()
    out["cat1_encoded"] = safe_label_transform(cat1_aux_encoder, out["cat1"].astype(str))
    out = out.drop(columns=["cat1"])
    return out.astype(float)

Xh_train = build_high_features(Xh_train_raw)
Xh_test = build_high_features(Xh_test_raw)

high_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

high_metrics = []
high_scores = {}
high_predictions = {}
for name, model in high_models.items():
    start = time.time()
    model.fit(Xh_train, yh_train)
    pred = model.predict(Xh_test)
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(Xh_test)[:, 1]
    else:
        score = pred
    precision, recall, f1, _ = precision_recall_fscore_support(yh_test, pred, average="binary", zero_division=0)
    row = {
        "model": name,
        "threshold_75_percentile": threshold,
        "AUC": roc_auc_score(yh_test, score),
        "Accuracy": accuracy_score(yh_test, pred),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Training Time": time.time() - start,
    }
    high_metrics.append(row)
    high_scores[name] = score
    high_predictions[name] = pred
    print(name, row)

high_metrics_df = pd.DataFrame(high_metrics).sort_values(["F1", "AUC"], ascending=False)
high_metrics_df.to_csv(MODEL_DIR / "high_buy_model_metrics.csv", index=False, encoding="utf-8-sig")

best_high_name = high_metrics_df.iloc[0]["model"]
print("high_buy ???????", best_high_name)
display(high_metrics_df)

In [ ]:
# 12. ?? 01???????
fig, ax = plt.subplots(figsize=(9, 5.4), facecolor="white")
plot_df = category_sales.sort_values("total_buy_mount", ascending=False)
bars = ax.bar(plot_df["cat1_name"], plot_df["total_buy_mount"], color=CHART_COLORS[:len(plot_df)], width=0.56)
ax.set_title("????????", fontsize=16, fontweight="bold")
ax.set_ylabel("????")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=10)
polish_axes(ax)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(CHART_DIR / "01_category_distribution.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 13. ?? 02???????
numeric_cols = ["buy_mount", "month", "weekday", "baby_age"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), facecolor="white")
for ax, col in zip(axes.ravel(), numeric_cols):
    data = model_df[col].replace([np.inf, -np.inf], np.nan).dropna()
    if col == "buy_mount":
        data = data.clip(upper=data.quantile(0.99))
    sns.histplot(data, bins=24, kde=True, ax=ax, color="#4E79A7", edgecolor="white")
    ax.set_title(col, fontsize=12, fontweight="bold")
    polish_axes(ax)
plt.tight_layout()
plt.savefig(CHART_DIR / "02_numeric_feature_distribution.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 14. ?? 03?????????
corr = X_train.corr()
fig, ax = plt.subplots(figsize=(10, 8), facecolor="white")
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.4, ax=ax, cbar_kws={"shrink": 0.78})
ax.set_title("???????????", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(CHART_DIR / "03_feature_correlation_heatmap.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 15. ?? 04?????? TOP10
feature_top = feature_importance.head(10).sort_values("importance")
fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
ax.barh(feature_top["feature"], feature_top["importance"], color="#8B5CF6")
ax.set_title("????????????? TOP10", fontsize=16, fontweight="bold")
ax.set_xlabel("Importance")
for i, v in enumerate(feature_top["importance"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=10)
polish_axes(ax, grid_axis="x")
plt.tight_layout()
plt.savefig(CHART_DIR / "04_feature_importance.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 16. ?? 05?cat1 ???????
metric_melt = cat1_metrics_df.melt(
    id_vars="model", value_vars=["Accuracy", "F1_weighted"], var_name="??", value_name="??"
)
metric_melt["??"] = metric_melt["??"].replace({"F1_weighted": "F1-score"})
fig, ax = plt.subplots(figsize=(11, 6.2), facecolor="white")
sns.barplot(data=metric_melt, x="model", y="??", hue="??", palette=["#3B82F6", "#22C55E"], ax=ax)
ax.set_ylim(0, 1.08)
ax.set_title("???????cat1 ????", fontsize=16, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Score")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=9, padding=2)
ax.text(0.02, 0.98, "???cat1 ??????????? Accuracy ? F1-score?AUC ??? high_buy ????????", transform=ax.transAxes, va="top", fontsize=10, color="#667085")
polish_axes(ax, grid_axis="y")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(CHART_DIR / "05_model_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 17. ?? 06?high_buy ROC ??
fpr, tpr, _ = roc_curve(yh_test, high_scores[best_high_name])
auc_value = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(8, 6.5), facecolor="white")
ax.plot(fpr, tpr, color="#8B5CF6", linewidth=2.6, label=f"{best_high_name} AUC={auc_value:.3f}")
ax.plot([0, 1], [0, 1], color="#98A2B3", linestyle="--", linewidth=1.2, label="????")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("???????? ROC ??", fontsize=16, fontweight="bold")
ax.legend(frameon=False)
polish_axes(ax)
plt.tight_layout()
plt.savefig(CHART_DIR / "06_roc_curve.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 18. ?? 07?high_buy ????
cm = confusion_matrix(yh_test, high_predictions[best_high_name])
fig, ax = plt.subplots(figsize=(7.6, 6.4), facecolor="white")
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", linewidths=0.5, cbar=False, ax=ax)
ax.set_title("????????????", fontsize=16, fontweight="bold")
ax.set_xlabel("????")
ax.set_ylabel("????")
ax.set_xticklabels(["????", "??????"], rotation=0)
ax.set_yticklabels(["????", "??????"], rotation=0)
plt.tight_layout()
plt.savefig(CHART_DIR / "07_confusion_matrix.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 19. ?? 08???????
prob_df = pd.DataFrame({
    "score": high_scores[best_high_name],
    "????": np.where(yh_test.values == 1, "??????", "??????"),
})
fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="white")
sns.histplot(data=prob_df, x="score", hue="????", bins=24, stat="density", common_norm=False, alpha=0.45, palette=["#3B82F6", "#EF476F"], ax=ax)
ax.set_title("????????????", fontsize=16, fontweight="bold")
ax.set_xlabel("????????????")
polish_axes(ax)
plt.tight_layout()
plt.savefig(CHART_DIR / "08_probability_distribution.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 20. ?? 09?????????
font_candidates = [
    "C:/Windows/Fonts/msyh.ttc", "C:/Windows/Fonts/simhei.ttf",
    "/usr/share/fonts/truetype/wqy/wqy-microhei.ttc",
    "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]
font_path = next((p for p in font_candidates if Path(p).exists()), None)

word_weights = merged_df.groupby("cat1_name")["buy_mount"].sum().sort_values(ascending=False).to_dict()
base = max(word_weights.values()) if word_weights else 1
frequencies = {word: float(weight) * 2.2 for word, weight in word_weights.items()}
for category, terms in CAT1_WORDS_EXTENDED.items():
    category_weight = float(word_weights.get(category, base * 0.35))
    frequencies[category] = max(float(frequencies.get(category, 0)), category_weight * 2.45)
    for idx, term in enumerate(terms):
        decay = max(0.18, 0.78 - idx * 0.035)
        frequencies[term] = max(float(frequencies.get(term, 0)), category_weight * decay)

mask_candidates = [
    PROJECT_ROOT / "assets" / "masks" / "mother_baby_wordcloud_mask_children.png",
    PROJECT_ROOT / "assets" / "masks" / "mother_baby_wordcloud_mask_stroller.png",
]
mask_path = next((p for p in mask_candidates if p.exists()), None)
mask = None
if mask_path:
    mask = np.array(Image.open(mask_path).convert("L").resize((1600, 1000), Image.LANCZOS))
    print("???? mask?", mask_path)
else:
    print("??? mask?????????")

palette = ["#3B82F6", "#22C55E", "#F59E0B", "#8B5CF6", "#06B6D4", "#EF476F"]
def color_func(*args, **kwargs):
    return palette[np.random.randint(0, len(palette))]

wc = WordCloud(
    font_path=font_path,
    width=1600,
    height=1000,
    background_color="white",
    max_words=90,
    max_font_size=215,
    min_font_size=7,
    prefer_horizontal=0.9,
    random_state=RANDOM_STATE,
    relative_scaling=0.5,
    collocations=False,
    repeat=True,
    margin=1,
    mask=mask,
    contour_width=0,
    color_func=color_func,
).generate_from_frequencies(frequencies)

wc.to_image().save(CHART_DIR / "09_wordcloud.png", dpi=(300, 300))
plt.figure(figsize=(12, 7.5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("??????????", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 21. ????????? TOP6??????????
# ?? TOP6 ?????
fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="white")
top6 = category_sales.sort_values("total_buy_mount", ascending=True).tail(6)
ax.barh(top6["cat1_name"], top6["total_buy_mount"], color=CHART_COLORS[:len(top6)])
ax.set_title("?????? TOP6", fontsize=16, fontweight="bold")
ax.set_xlabel("????")
for i, v in enumerate(top6["total_buy_mount"]):
    ax.text(v, i, f" {int(v):,}", va="center", fontsize=10)
polish_axes(ax, grid_axis="x")
plt.tight_layout()
plt.savefig(CHART_DIR / "10_category_top6.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# ?????????
fig, ax1 = plt.subplots(figsize=(12, 5.8), facecolor="white")
ax2 = ax1.twinx()
ax1.bar(monthly_sales["year_month"], monthly_sales["total_buy_mount"], color="#FF8FAB", alpha=0.78, label="????")
ax2.plot(monthly_sales["year_month"], monthly_sales["trade_count"], color="#3B82F6", marker="o", linewidth=2.2, label="???")
ax1.set_title("??????", fontsize=16, fontweight="bold")
ax1.set_ylabel("????")
ax2.set_ylabel("???")
ax1.tick_params(axis="x", rotation=30, labelsize=9)
polish_axes(ax1)
ax2.grid(False)
plt.tight_layout()
plt.savefig(CHART_DIR / "11_monthly_trend.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# ?????????
weekday_names = {1: "??", 2: "??", 3: "??", 4: "??", 5: "??", 6: "??", 7: "??"}
weekday_plot = weekday_sales.copy()
weekday_plot["weekday_name"] = weekday_plot["weekday"].map(weekday_names)
fig, ax1 = plt.subplots(figsize=(9, 5.2), facecolor="white")
ax2 = ax1.twinx()
ax1.bar(weekday_plot["weekday_name"], weekday_plot["total_buy_mount"], color="#22C55E", alpha=0.75, label="????")
ax2.plot(weekday_plot["weekday_name"], weekday_plot["trade_count"], color="#3B82F6", marker="o", linewidth=2.2, label="???")
ax1.set_title("???????", fontsize=16, fontweight="bold")
ax1.set_ylabel("????")
ax2.set_ylabel("???")
polish_axes(ax1)
ax2.grid(False)
plt.tight_layout()
plt.savefig(CHART_DIR / "12_weekday_activity.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 22. ??????
print("??????", PROCESSED_DIR / "merged_mother_baby_data.csv")
print("??????", TABLE_DIR)
print("???????", MODEL_DIR)
print("?????", CHART_DIR)

for p in sorted(CHART_DIR.glob("*.png")):
    print(p.name, f"{p.stat().st_size/1024:.1f} KB")

## ??????

1. ?????????????????
2. ??????????????????????????????????
3. ????????`cat_id_encoded`?`property_freq`?`auction_id_freq`?`cat_id_freq`???????????
4. ???? `cat1` ??????????????????????????????
5. ????? `high_buy`??? `buy_mount`????????????????
6. ?????????????????????????ROC???????????????